# Poverty Incidence Profiling
Table: `pondolens.dev_bronze.poverty_incidence`  
Baseline: 644 rows

**Goal:** identify records suitable for **Q3: How does regional government spending compare with population and economic conditions?** Poverty incidence provides context on regional needs. 

**Row meaning:** geographic area × year × reference unit (families or population). The file includes national, regional, and province/district records.

## 1. Structure and coverage
Inspect the schema, sample rows, years, and measures. Keep source metadata for traceability.

In [0]:
%sql
DESCRIBE TABLE pondolens.dev_bronze.poverty_incidence;

In [0]:
%sql
SELECT * FROM pondolens.dev_bronze.poverty_incidence
LIMIT 10;

In [0]:
%sql
SELECT year, reference_unit,
    COUNT(*) AS row_count,
    COUNT(DISTINCT region) AS geographic_labels,
    COUNT_IF(UPPER(TRIM(region)) = 'PHILIPPINES') AS national_rows,
    COUNT_IF(UPPER(TRIM(region)) <> 'PHILIPPINES'
        AND NULLIF(TRIM(province), '') IS NULL) AS regional_rows,
    COUNT_IF(NULLIF(TRIM(province), '') IS NOT NULL) AS province_or_district_rows
FROM pondolens.dev_bronze.poverty_incidence
GROUP BY year, reference_unit
ORDER BY year, reference_unit;

## 2. Data quality and numeric parsing
Check required fields and parse all six numeric measures. Blank province is expected for regional/national records. Blank region_name is checked separately because region is the primary label in this source. A `-` is a source marker, not zero.

In [0]:
%sql
SELECT COUNT(*) AS total_rows,
    COUNT_IF(NULLIF(TRIM(region), '') IS NULL) AS missing_region,
    COUNT_IF(NULLIF(TRIM(region_name), '') IS NULL) AS missing_region_name,
    COUNT_IF(NULLIF(TRIM(reference_unit), '') IS NULL) AS missing_reference_unit,
    COUNT_IF(TRY_CAST(year AS INT) IS NULL) AS missing_or_invalid_year,
    COUNT_IF(NULLIF(TRIM(source_name), '') IS NULL) AS missing_source_name,
    COUNT_IF(NULLIF(TRIM(source_page), '') IS NULL) AS missing_source_page,
    COUNT_IF(TRY_CAST(source_row AS INT) IS NULL) AS missing_or_invalid_source_row,
    COUNT_IF(NULLIF(LOWER(TRIM(CAST(_rescued_data AS STRING))), 'null') IS NOT NULL
        AND TRIM(CAST(_rescued_data AS STRING)) <> '') AS rescued_rows
FROM pondolens.dev_bronze.poverty_incidence;

In [0]:
%sql
WITH numeric_fields AS (
SELECT 'annual_poverty_threshold_per_capita' AS field, CAST(annual_poverty_threshold_per_capita AS STRING) AS raw_value FROM pondolens.dev_bronze.poverty_incidence
UNION ALL
SELECT 'poverty_incidence_estimate' AS field, CAST(poverty_incidence_estimate AS STRING) AS raw_value FROM pondolens.dev_bronze.poverty_incidence
UNION ALL
SELECT 'poverty_incidence_coeff_var' AS field, CAST(poverty_incidence_coeff_var AS STRING) AS raw_value FROM pondolens.dev_bronze.poverty_incidence
UNION ALL
SELECT 'poverty_incidence_standard_error' AS field, CAST(poverty_incidence_standard_error AS STRING) AS raw_value FROM pondolens.dev_bronze.poverty_incidence
UNION ALL
SELECT 'poverty_incidence_confidence_lower' AS field, CAST(poverty_incidence_confidence_lower AS STRING) AS raw_value FROM pondolens.dev_bronze.poverty_incidence
UNION ALL
SELECT 'poverty_incidence_confidence_upper' AS field, CAST(poverty_incidence_confidence_upper AS STRING) AS raw_value FROM pondolens.dev_bronze.poverty_incidence
), checked AS (
    SELECT field, TRIM(raw_value) AS raw_value,
        TRY_CAST(TRIM(raw_value) AS DECIMAL(28,12)) AS numeric_value
    FROM numeric_fields
)
SELECT field,
    COUNT_IF(NULLIF(raw_value, '') IS NULL) AS missing_values,
    COUNT_IF(raw_value = '-') AS dash_markers,
    COUNT_IF(NULLIF(raw_value, '') IS NOT NULL AND numeric_value IS NULL) AS non_numeric_values,
    MIN(numeric_value) AS minimum_value,
    MAX(numeric_value) AS maximum_value
FROM checked
GROUP BY field
ORDER BY field;

In [0]:
%sql
WITH checked AS (
    SELECT TRY_CAST(annual_poverty_threshold_per_capita AS DECIMAL(28,12)) AS threshold_value,
        TRY_CAST(poverty_incidence_estimate AS DECIMAL(28,12)) AS estimate,
        TRY_CAST(poverty_incidence_coeff_var AS DECIMAL(28,12)) AS coeff_var,
        TRY_CAST(poverty_incidence_standard_error AS DECIMAL(28,12)) AS standard_error,
        TRY_CAST(poverty_incidence_confidence_lower AS DECIMAL(28,12)) AS lower_bound,
        TRY_CAST(poverty_incidence_confidence_upper AS DECIMAL(28,12)) AS upper_bound
    FROM pondolens.dev_bronze.poverty_incidence
)
SELECT COUNT_IF(threshold_value <= 0) AS non_positive_threshold,
    COUNT_IF(estimate < 0 OR estimate > 100) AS incidence_outside_0_100,
    COUNT_IF(coeff_var < 0 OR standard_error < 0) AS negative_uncertainty,
    COUNT_IF(lower_bound < 0 OR upper_bound > 100
        OR lower_bound > estimate OR estimate > upper_bound) AS invalid_confidence_interval
FROM checked;

## 3. Check for duplicate source records
**Key: region + province + year + reference_unit.** This checks whether the same area, year and measure appears more than once in the source. `province` also contains district labels; blank/null province values are treated as the same key value.

Zero duplicate key groups means no repeated source-record keys were found.

In [0]:
%sql
WITH key_counts AS (
    SELECT UPPER(TRIM(region)) AS region_key,
        COALESCE(NULLIF(UPPER(TRIM(province)), ''), '') AS province_key,
        TRY_CAST(year AS INT) AS year_key,
        LOWER(TRIM(reference_unit)) AS measure_key,
        COUNT(*) AS row_count
    FROM pondolens.dev_bronze.poverty_incidence
    GROUP BY UPPER(TRIM(region)), COALESCE(NULLIF(UPPER(TRIM(province)), ''), ''),
        TRY_CAST(year AS INT), LOWER(TRIM(reference_unit))
)
SELECT COUNT_IF(row_count > 1) AS duplicate_key_groups,
    COALESCE(SUM(CASE WHEN row_count > 1 THEN row_count ELSE 0 END), 0) AS rows_in_duplicate_groups
FROM key_counts;

## 4. Q3 candidates and duplicate join checks
**Join key: region + year, after selecting regional totals and one reference_unit.** This checks whether joining poverty data to regional spending would attach more than one poverty row to the same region/year.

Regional totals have blank province and exclude PHILIPPINES. Each region/year currently has two rows: one family rate and one population rate. These are different measures, not duplicate source records. For Q3, propose filtering to `Poverty incidence among population`, leaving one poverty row per region/year.

The first query compares key counts before and after that filter. The second lists the selected records. Spending-side uniqueness and matching regional boundaries still need checking.

In [0]:
%sql
WITH regional AS (
    SELECT UPPER(TRIM(region)) AS region_key, TRY_CAST(year AS INT) AS year_key,
        LOWER(TRIM(reference_unit)) AS measure_key
    FROM pondolens.dev_bronze.poverty_incidence
    WHERE UPPER(TRIM(region)) <> 'PHILIPPINES'
        AND NULLIF(TRIM(province), '') IS NULL
), key_counts AS (
    SELECT region_key, year_key, COUNT(*) AS all_measure_rows,
        COUNT_IF(measure_key = 'poverty incidence among population') AS population_rows
    FROM regional
    GROUP BY region_key, year_key
)
SELECT COUNT(*) AS region_year_keys,
    COUNT_IF(all_measure_rows > 1) AS repeated_keys_before_measure_filter,
    COUNT_IF(population_rows > 1) AS duplicate_keys_after_measure_filter,
    COUNT_IF(population_rows = 0) AS keys_missing_population_measure,
    SUM(population_rows) AS population_candidate_rows
FROM key_counts;

In [0]:
%sql
SELECT region, region_name, TRY_CAST(year AS INT) AS year,
    reference_unit,
    TRY_CAST(poverty_incidence_estimate AS DECIMAL(18,6)) AS poverty_incidence_pct,
    TRY_CAST(annual_poverty_threshold_per_capita AS DECIMAL(18,2)) AS annual_threshold_per_capita,
    poverty_incidence_confidence_lower, poverty_incidence_confidence_upper,
    notes, source_name, source_page, source_row
FROM pondolens.dev_bronze.poverty_incidence
WHERE UPPER(TRIM(region)) <> 'PHILIPPINES'
    AND NULLIF(TRIM(province), '') IS NULL
    AND LOWER(TRIM(reference_unit)) = 'poverty incidence among population'
ORDER BY year, region;

## 5. Findings and proposed Silver treatments

| Check | Finding | Proposed Silver treatment |
|---|---|---|
| Coverage | 644 rows: 6 national, 108 regional, 530 province/district. Years: 2021, 2023, 2025; 18 regions including NIR. | Keep all records. Add a geographic-level label: national, regional, or province/district. For Q3, use only the regional rows with blank province. |
| Data types | All columns are STRING except _ingested_at, which is TIMESTAMP. _source_modified_at is STRING; source_page contains tab1/tab2. | Cast year/source_row to INT, threshold to DECIMAL(18,2), and the five poverty statistics to DECIMAL(18,6). Keep source_page and other metadata as STRING for now; keep _ingested_at as TIMESTAMP. |
| Required fields | No missing region, reference_unit, source_name or source_page. year and source_row parse as integers. | Keep these values required in Silver. |
| Rescued data | _rescued_data is null throughout the export: no values were captured there as ingestion exceptions. This does not prove all data is correct. | Keep _rescued_data for now and review any populated values on later loads. |
| Numeric values | Threshold, estimate, standard error and confidence bounds parse; range/interval checks pass. Coefficient of variation has 8 `-` markers. | Convert these 8 markers to null or zero depending on source review; retain the original marker until confirmed. |
| Geographic labels | region_name is blank in 36 MIMAROPA rows and 6 PHILIPPINES rows. | Need review. Initial proposal is region_name = MIMAROPA for region = MIMAROPA Region, and region_name = Philippines for region = PHILIPPINES. Mark Philippines as national and exclude it from regional joins. |
| Source-record key | No duplicates for region + province + year + reference_unit. | Recheck this full key in Silver after cleaning. |
| Q3 join key | Regional region + year has 54 keys with two measures each. Filtering reference_unit to population leaves 54 rows with unique region + year keys. | Use one agreed measure for Q3. After mapping regions to shared codes, confirm one poverty row per region_code + year so the join does not repeat spending rows. |
| Source metadata | Source fields, notes, ingestion metadata, hashes and run ID are present; _source_sheet is null. | Keep source and ingestion metadata for now, including footnotes and null _source_sheet. |

Data findings describe the supplied CSV; types are confirmed from the supplied Databricks schema. Confirm live data results after running the notebook.

## 6. Assumptions and open checks

| Question | Status |
|---|---|
| Does blank province identify a regional/national total? | Need further review and check with original source. |
| Are the proposed MIMAROPA and Philippines labels suitable? | Pending team review and mapping to shared regional codes. Philippines is a national total. |
| Should Q3 use population poverty incidence? | Proposed; confirm with the team before finalizing measure. |
| What do the 8 coefficient-of-variation `-` markers mean? | Pending source review before converting to null or zero. |
| Are regional boundaries and join keys compatible across datasets? | Pending NIR/boundary review, spending-key checks and unmatched-region checks. Use matching available years only; poverty data has no 2022 or 2024 rows. |

## 7. Semantic notes

- Family and population incidence have different denominators. Keep them separate; do not sum poverty percentages or combine national, regional and province/district records.
- Annual poverty threshold per capita is a monetary threshold, not a poverty rate or government spending.
- Keep confidence bounds and source footnotes with the estimates. A `-` is not yet confirmed zero or null.
- Poverty incidence gives Q3 context on regional needs; a spending comparison does not establish that spending caused poverty changes.

## 8. Conclusion

- Required-field, numeric-range and source-record-key checks passed. Eight coefficient-of-variation `-` markers and blank MIMAROPA/Philippines names still need review.
- In Silver, cast year/source_row and numeric measures to the proposed types, add geographic-level labels, and keep source metadata and the ingestion TIMESTAMP.
- Q3 has 54 regional population-incidence candidates for 2021, 2023 and 2025. Confirm the measure, source notes and regional boundaries.
- Data is currently for 2021, 2023, adn 2025 due to lack of yearly annual data. Confirm if indicator can proceed to be used even with the gaps in the years.